In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

In [2]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
import math

backend = AerSimulator()

In [3]:
# SHARED UTILITY 
# Quantum random bit generator. The protocol requires several random
# choices (Alice's bits, Alice's bases, Bob's bases, sample selection
# for the eavesdropping check, and Eve's bases in the attacker version).
# We obtain randomness by measuring |+> in the computational basis:
# applying H to |0> gives (|0> + |1>)/sqrt(2), and measuring in Z
# yields 0 or 1 with equal probability.

# To amortise the simulator cost we build one k-qubit Hadamard circuit
# at a time and read k independent random bits from a single shot.
# We chunk to keep individual circuits small.

_CHUNK = 24  # bits per circuit

def _random_chunk(k):
    qc = QuantumCircuit(k, k)
    qc.h(range(k))
    qc.measure(range(k), range(k))
    counts = backend.run(qc, shots=1).result().get_counts()
    bitstring = next(iter(counts))
    # Qiskit prints the highest-index qubit on the left; reverse so
    # index i in the returned list corresponds to qubit i.
    return [int(b) for b in reversed(bitstring)]

def quantum_random_bits(n):
    """Return a list of n random bits from measuring |+> states in Z."""
    bits = []
    remaining = n
    while remaining > 0:
        k = min(_CHUNK, remaining)
        bits.extend(_random_chunk(k))
        remaining -= k
    return bits

def quantum_random_bit():
    return quantum_random_bits(1)[0]

# Sanity check: should be roughly half 0s and half 1s.
sample = quantum_random_bits(1000)
print(f"Sample of 1000 quantum random bits: {sum(sample)} ones, {1000 - sum(sample)} zeros")

Sample of 1000 quantum random bits: 491 ones, 509 zeros


In [4]:
# ALICE'S CODE
# For each transmission Alice picks a random bit and a random basis:
#   basis 0 = Z (computational): bit 0 -> |0>,  bit 1 -> |1>
#   basis 1 = X (diagonal):      bit 0 -> |+>,  bit 1 -> |->
# She returns a 1-qubit circuit that prepares that state.

def alice_prepare(bit, basis):
    """Return a 1-qubit circuit preparing Alice's chosen state."""
    qc = QuantumCircuit(1, 1)
    if bit == 1:
        qc.x(0)
    if basis == 1:
        qc.h(0)
    return qc

In [ ]:
# BOB'S CODE
# To measure in basis b, Bob applies H if b = 1 (rotating the X-basis
# back to the computational basis) and then measures in Z. If his
# basis matches Alice's, his result equals Alice's bit; otherwise the
# outcome is uniformly random.

def bob_measure(qc, basis):
    """Append Bob's measurement to qc, run it, return his classical bit."""
    if basis == 1:
        qc.h(0)
    qc.measure(0, 0)
    counts = backend.run(qc, shots=1).result().get_counts()
    bitstring = next(iter(counts))
    return int(bitstring[0])

In [ ]:
def transmit_qubit(alice_bit, alice_basis, bob_basis):
    """Alice prepares, Bob measures. Returns Bob's recorded bit."""
    qc = alice_prepare(alice_bit, alice_basis)   # ALICE
    # ---- quantum channel (noiseless, no Eve) ----
    bob_bit = bob_measure(qc, bob_basis)         # BOB
    return bob_bit

In [7]:
# SHARED HELPER
# Sifting step: Alice and Bob publicly announce their bases (but not
# their bits) and keep only the positions where the bases agree.

def sift(alice_bits, alice_bases, bob_bits, bob_bases):
    matching = [i for i in range(len(alice_bases)) if alice_bases[i] == bob_bases[i]]
    a_sift = [alice_bits[i] for i in matching]
    b_sift = [bob_bits[i]   for i in matching]
    return a_sift, b_sift, matching

In [8]:
N = 200

# Alice's preparation choices
alice_bits  = quantum_random_bits(N)
alice_bases = quantum_random_bits(N)

# Bob's measurement choices 
bob_bases   = quantum_random_bits(N)

# Sequential transmission loop 
bob_results = []
for i in range(N):
    bob_results.append(transmit_qubit(alice_bits[i], alice_bases[i], bob_bases[i]))

# Sifting: keep positions where bases matched 
sifted_alice, sifted_bob, matching = sift(alice_bits, alice_bases, bob_results, bob_bases)

print(f"Qubits sent:                {N}")
print(f"Matching-basis positions:   {len(matching)}")
print(f"Sifted key length:          {len(sifted_alice)}")
print()
print(f"Alice sifted key: {''.join(str(b) for b in sifted_alice)}")
print(f"Bob   sifted key: {''.join(str(b) for b in sifted_bob)}")
print()
agree = sum(1 for a, b in zip(sifted_alice, sifted_bob) if a == b)
print(f"Agreement: {agree}/{len(sifted_alice)}")
assert sifted_alice == sifted_bob, "Sifted keys must agree on a noiseless channel without Eve."
print("\nNo attacker, no noise => sifted keys agree perfectly.")

Qubits sent:                200
Matching-basis positions:   81
Sifted key length:          81

Alice sifted key: 110111110100110011010101011011101011001010001001100010100101111100110110101111010
Bob   sifted key: 110111110100110011010101011011101011001010001001100010100101111100110110101111010

Agreement: 81/81

No attacker, no noise => sifted keys agree perfectly.


In [9]:
TRIALS = 5
N      = 200

for trial in range(TRIALS):
    a_bits  = quantum_random_bits(N)
    a_bases = quantum_random_bits(N)
    b_bases = quantum_random_bits(N)
    b_bits  = [transmit_qubit(a_bits[i], a_bases[i], b_bases[i]) for i in range(N)]
    sa, sb, _ = sift(a_bits, a_bases, b_bits, b_bases)
    print(f"trial {trial}:  sifted_len={len(sa):3d}  keys_match={sa == sb}")

trial 0:  sifted_len=102  keys_match=True
trial 1:  sifted_len=100  keys_match=True
trial 2:  sifted_len= 91  keys_match=True
trial 3:  sifted_len= 99  keys_match=True
trial 4:  sifted_len=104  keys_match=True
